### Setup & load data

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

EXTERNAL = PROJECT_ROOT / "data" / "external"
PROCESSED = PROJECT_ROOT / "data" / "processed"

EXTERNAL.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

metadata_path = EXTERNAL / "circuit_metadata.csv"
driver_path = PROCESSED / "driver_race_with_constructor_features.csv"

circuit_metadata = pd.read_csv(metadata_path)
driver_race = pd.read_csv(driver_path)

print("Circuit metadata:", circuit_metadata.shape)
print("Driver-race with constructor features:", driver_race.shape)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Circuit metadata: (35, 22)
Driver-race with constructor features: (6911, 111)


### Validate required columns

In [2]:
required_metadata_cols = [
    "circuit_id",
    "circuit_name",
    "circuit_type",
    "lap_length_km",
    "altitude_m",
    "downforce_score",
    "speed_score",
    "technical_score",
    "overtaking_score",
    "downforce_level",
    "speed_profile",
    "technical_level",
    "overtaking_level",
    "lap_length_bucket",
    "altitude_bucket",
    "metadata_confidence",
    "lap_length_source",
    "altitude_source",
    "classification_source",
    "notes"
]

required_driver_cols = [
    "season",
    "round",
    "race_id",
    "driver_race_id",
    "driver_id",
    "driver_name",
    "driver_nationality",
    "constructor_id",
    "constructor_name",
    "constructor_nationality",
    "circuit_id",
    "circuit_country"
]

missing_metadata = [col for col in required_metadata_cols if col not in circuit_metadata.columns]
missing_driver = [col for col in required_driver_cols if col not in driver_race.columns]

print("Missing metadata columns:", missing_metadata)
print("Missing driver-race columns:", missing_driver)

if missing_metadata:
    raise KeyError(f"Missing metadata columns: {missing_metadata}")

if missing_driver:
    raise KeyError(f"Missing driver-race columns: {missing_driver}")

Missing metadata columns: []
Missing driver-race columns: []


### Clean metadata types and labels

In [3]:
# Clean numeric metadata fields
numeric_metadata_cols = [
    "lap_length_km",
    "altitude_m",
    "downforce_score",
    "speed_score",
    "technical_score",
    "overtaking_score"
]

for col in numeric_metadata_cols:
    circuit_metadata[col] = pd.to_numeric(circuit_metadata[col], errors="coerce")

# Standardize categorical text
text_cols = [
    "circuit_id",
    "circuit_type",
    "downforce_level",
    "speed_profile",
    "technical_level",
    "overtaking_level",
    "lap_length_bucket",
    "altitude_bucket",
    "metadata_confidence"
]

for col in text_cols:
    circuit_metadata[col] = circuit_metadata[col].astype(str).str.strip()

# Prevent duplicate circuit metadata rows
circuit_metadata = circuit_metadata.drop_duplicates(
    subset=["circuit_id"],
    keep="last"
).reset_index(drop=True)

print("Circuit metadata after cleaning:", circuit_metadata.shape)
print("Duplicate circuit IDs:", circuit_metadata.duplicated("circuit_id").sum())

circuit_metadata.head()

Circuit metadata after cleaning: (35, 22)
Duplicate circuit IDs: 0


,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_type,lap_length_km,altitude_m,downforce_score,speed_score,technical_score,...,speed_profile,technical_level,overtaking_level,lap_length_bucket,altitude_bucket,metadata_confidence,lap_length_source,altitude_source,classification_source,notes
0,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,hybrid,5.278,7,6,7,6,...,balanced,medium,moderate,medium,low,medium,https://www.formula1.com/en/racing/2025/australia,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Parkland/semi-street circuit; medium-high spee...
1,americas,Circuit of the Americas,Austin,USA,permanent,5.513,150,6,7,8,...,balanced,high,moderate,long,low,high,https://en.wikipedia.org/wiki/Circuit_of_the_A...,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with elevation change and mi...
2,bahrain,Bahrain International Circuit,Sakhir,Bahrain,permanent,5.412,7,5,7,6,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...
3,baku,Baku City Circuit,Baku,Azerbaijan,street,6.003,-20,3,10,7,...,high_speed,medium,moderate,long,low,medium,https://www.formula1.com/en/racing/2025/azerba...,estimated from Baku coastal elevation references,analyst-coded estimate; not an official F1 num...,Street circuit with very long full-throttle se...
4,buddh,Buddh International Circuit,Uttar Pradesh,India,permanent,5.125,195,6,8,7,...,high_speed,medium,moderate,medium,low,medium,https://en.wikipedia.org/wiki/Buddh_Internatio...,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent Tilke circuit with long straights an...


### Metadata validation

In [4]:
print("=" * 80)
print("VALIDATION — CIRCUIT METADATA")
print("=" * 80)

print("\nDATASET SHAPE")
print("-" * 80)
print("Rows:", len(circuit_metadata))
print("Columns:", circuit_metadata.shape[1])
print("Unique circuits:", circuit_metadata["circuit_id"].nunique())

print("\nMISSINGNESS")
print("-" * 80)
print(circuit_metadata[required_metadata_cols].isna().sum())

print("\nVALID CATEGORY CHECKS")
print("-" * 80)

valid_values = {
    "circuit_type": {"street", "permanent", "hybrid"},
    "downforce_level": {"low", "medium", "high"},
    "speed_profile": {"low_speed", "balanced", "high_speed"},
    "technical_level": {"low", "medium", "high"},
    "overtaking_level": {"difficult", "moderate", "easy"},
    "lap_length_bucket": {"short", "medium", "long"},
    "altitude_bucket": {"low", "medium", "high"},
    "metadata_confidence": {"low", "medium", "high"}
}

issues = []

for col, allowed in valid_values.items():
    actual = set(circuit_metadata[col].dropna().unique())
    invalid = actual - allowed
    print(f"{col}: invalid values = {invalid}")
    if invalid:
        issues.append(f"{col} has invalid values: {invalid}")

print("\nSCORE RANGE CHECKS")
print("-" * 80)

for col in ["downforce_score", "speed_score", "technical_score", "overtaking_score"]:
    col_min = circuit_metadata[col].min()
    col_max = circuit_metadata[col].max()
    print(f"{col}: min={col_min}, max={col_max}")

    if circuit_metadata[col].dropna().lt(1).any() or circuit_metadata[col].dropna().gt(10).any():
        issues.append(f"{col} has values outside 1–10.")

if circuit_metadata["lap_length_km"].dropna().le(0).any():
    issues.append("lap_length_km has non-positive values.")

if circuit_metadata["circuit_id"].duplicated().sum() != 0:
    issues.append("Duplicate circuit_id rows exist.")

if issues:
    print("\nVERDICT: REVIEW NEEDED")
    for issue in issues:
        print("-", issue)
else:
    print("\nVERDICT: PASS — circuit metadata is technically valid.")

VALIDATION — CIRCUIT METADATA

DATASET SHAPE
--------------------------------------------------------------------------------
Rows: 35
Columns: 22
Unique circuits: 35

MISSINGNESS
--------------------------------------------------------------------------------
circuit_id               0
circuit_name             0
circuit_type             0
lap_length_km            0
altitude_m               0
downforce_score          0
speed_score              0
technical_score          0
overtaking_score         0
downforce_level          0
speed_profile            0
technical_level          0
overtaking_level         0
lap_length_bucket        0
altitude_bucket          0
metadata_confidence      0
lap_length_source        0
altitude_source          0
classification_source    0
notes                    0
dtype: int64

VALID CATEGORY CHECKS
--------------------------------------------------------------------------------
circuit_type: invalid values = set()
downforce_level: invalid values = set()
speed

### Check circuit coverage against driver-race data

In [5]:
driver_circuits = set(driver_race["circuit_id"].dropna().unique())
metadata_circuits = set(circuit_metadata["circuit_id"].dropna().unique())

missing_from_metadata = sorted(driver_circuits - metadata_circuits)
extra_metadata = sorted(metadata_circuits - driver_circuits)

print("=" * 80)
print("CIRCUIT COVERAGE CHECK")
print("=" * 80)

print("Circuits in driver-race data:", len(driver_circuits))
print("Circuits in metadata:", len(metadata_circuits))
print("Missing from metadata:", len(missing_from_metadata))
print("Extra metadata circuits:", len(extra_metadata))

if missing_from_metadata:
    print("\nMissing circuits:")
    print(missing_from_metadata)

if extra_metadata:
    print("\nExtra metadata circuits:")
    print(extra_metadata)

if missing_from_metadata:
    raise ValueError("Some circuits in driver-race data are missing from circuit_metadata.csv.")

CIRCUIT COVERAGE CHECK
Circuits in driver-race data: 35
Circuits in metadata: 35
Missing from metadata: 0
Extra metadata circuits: 0


### Merge circuit metadata

In [6]:
rows_before_merge = len(driver_race)

driver_race_with_circuit = driver_race.merge(
    circuit_metadata,
    on="circuit_id",
    how="left",
    suffixes=("", "_metadata")
)

print("Rows before merge:", rows_before_merge)
print("Rows after merge:", len(driver_race_with_circuit))
print("Duplicate driver-race rows:",
      driver_race_with_circuit.duplicated(["season", "round", "driver_id"]).sum())

driver_race_with_circuit.head()

Rows before merge: 6911
Rows after merge: 6911
Duplicate driver-race rows: 0


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,speed_profile,technical_level,overtaking_level,lap_length_bucket,altitude_bucket,metadata_confidence,lap_length_source,altitude_source,classification_source,notes
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,liuzzi,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,sutil,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,bruno_senna,...,balanced,medium,easy,medium,low,high,https://www.formula1.com/en/racing/2025/bahrain,estimated from circuit/locality elevation refe...,analyst-coded estimate; not an official F1 num...,Permanent circuit with long straights heavy br...


### Create home race flags

In [7]:
# Normalize country/nationality text for practical matching
driver_country = driver_race_with_circuit["driver_nationality"].astype(str).str.lower().str.strip()
constructor_country = driver_race_with_circuit["constructor_nationality"].astype(str).str.lower().str.strip()
race_country = driver_race_with_circuit["circuit_country"].astype(str).str.lower().str.strip()

# Practical overrides for common nationality/country naming mismatches
country_aliases = {
    "british": "uk",
    "american": "usa",
    "dutch": "netherlands",
    "mexican": "mexico",
    "spanish": "spain",
    "french": "france",
    "german": "germany",
    "italian": "italy",
    "australian": "australia",
    "canadian": "canada",
    "japanese": "japan",
    "brazilian": "brazil",
    "monegasque": "monaco",
    "thai": "thailand",
    "chinese": "china",
    "finnish": "finland",
    "danish": "denmark",
    "swedish": "sweden",
    "russian": "russia",
    "polish": "poland",
    "indian": "india",
    "venezuelan": "venezuela",
    "indonesian": "indonesia",
    "new zealander": "new zealand",
    "argentine": "argentina",
    "french-italian": "france"
}

driver_country_mapped = driver_country.replace(country_aliases)
constructor_country_mapped = constructor_country.replace(country_aliases)

driver_race_with_circuit["home_race_driver"] = (
    driver_country_mapped == race_country
).astype(int)

driver_race_with_circuit["home_race_constructor"] = (
    constructor_country_mapped == race_country
).astype(int)

print("Driver home race rows:", driver_race_with_circuit["home_race_driver"].sum())
print("Constructor home race rows:", driver_race_with_circuit["home_race_constructor"].sum())

display(
    driver_race_with_circuit[
        (driver_race_with_circuit["home_race_driver"] == 1)
        | (driver_race_with_circuit["home_race_constructor"] == 1)
    ][[
        "season",
        "round",
        "race_name",
        "driver_name",
        "driver_nationality",
        "constructor_name",
        "constructor_nationality",
        "circuit_country",
        "home_race_driver",
        "home_race_constructor"
    ]].head(40)
)

Driver home race rows: 232
Constructor home race rows: 265


,season,round,race_name,driver_name,driver_nationality,constructor_name,constructor_nationality,circuit_country,home_race_driver,home_race_constructor
37,2010,2,Australian Grand Prix,Mark Webber,Australian,Red Bull,Austrian,Australia,1,0
96,2010,5,Spanish Grand Prix,Fernando Alonso,Spanish,Ferrari,Italian,Spain,1,0
100,2010,5,Spanish Grand Prix,Bruno Senna,Brazilian,HRT,Spanish,Spain,0,1
101,2010,5,Spanish Grand Prix,Karun Chandhok,Indian,HRT,Spanish,Spain,0,1
113,2010,5,Spanish Grand Prix,Pedro de la Rosa,Spanish,Sauber,Swiss,Spain,1,0
114,2010,5,Spanish Grand Prix,Jaime Alguersuari,Spanish,Toro Rosso,Italian,Spain,1,0
192,2010,9,European Grand Prix,Fernando Alonso,Spanish,Ferrari,Italian,Spain,1,0
196,2010,9,European Grand Prix,Bruno Senna,Brazilian,HRT,Spanish,Spain,0,1
197,2010,9,European Grand Prix,Karun Chandhok,Indian,HRT,Spanish,Spain,0,1
209,2010,9,European Grand Prix,Pedro de la Rosa,Spanish,Sauber,Swiss,Spain,1,0


### Technical validation

In [8]:
print("=" * 80)
print("VALIDATION — DRIVER-RACE WITH CIRCUIT FEATURES")
print("=" * 80)

print("\nDATASET SHAPE")
print("-" * 80)
print("Input rows:", len(driver_race))
print("Output rows:", len(driver_race_with_circuit))
print("Output columns:", driver_race_with_circuit.shape[1])

print("\nROW INTEGRITY")
print("-" * 80)
duplicate_driver_races = driver_race_with_circuit.duplicated(["season", "round", "driver_id"]).sum()
print("Driver-race row count preserved:", len(driver_race_with_circuit) == len(driver_race))
print("Duplicate driver-race rows:", duplicate_driver_races)

print("\nCIRCUIT FEATURE MISSINGNESS")
print("-" * 80)

circuit_feature_cols = [
    "circuit_type",
    "lap_length_km",
    "altitude_m",
    "downforce_score",
    "speed_score",
    "technical_score",
    "overtaking_score",
    "downforce_level",
    "speed_profile",
    "technical_level",
    "overtaking_level",
    "lap_length_bucket",
    "altitude_bucket"
]

print(driver_race_with_circuit[circuit_feature_cols].isna().sum())

print("\nHOME RACE FLAGS")
print("-" * 80)
print("home_race_driver values:")
print(driver_race_with_circuit["home_race_driver"].value_counts(dropna=False))
print("\nhome_race_constructor values:")
print(driver_race_with_circuit["home_race_constructor"].value_counts(dropna=False))

issues = []

if len(driver_race_with_circuit) != len(driver_race):
    issues.append("Driver-race row count changed after circuit merge.")

if duplicate_driver_races != 0:
    issues.append("Duplicate driver-race rows exist after circuit merge.")

if driver_race_with_circuit[circuit_feature_cols].isna().sum().sum() != 0:
    issues.append("Some circuit features are missing after merge.")

if not set(driver_race_with_circuit["home_race_driver"].unique()).issubset({0, 1}):
    issues.append("home_race_driver has values outside 0/1.")

if not set(driver_race_with_circuit["home_race_constructor"].unique()).issubset({0, 1}):
    issues.append("home_race_constructor has values outside 0/1.")

if issues:
    print("\nVERDICT: REVIEW NEEDED")
    for issue in issues:
        print("-", issue)
else:
    print("\nVERDICT: PASS — driver-race circuit features are technically valid.")

VALIDATION — DRIVER-RACE WITH CIRCUIT FEATURES

DATASET SHAPE
--------------------------------------------------------------------------------
Input rows: 6911
Output rows: 6911
Output columns: 134

ROW INTEGRITY
--------------------------------------------------------------------------------
Driver-race row count preserved: True
Duplicate driver-race rows: 0

CIRCUIT FEATURE MISSINGNESS
--------------------------------------------------------------------------------
circuit_type         0
lap_length_km        0
altitude_m           0
downforce_score      0
speed_score          0
technical_score      0
overtaking_score     0
downforce_level      0
speed_profile        0
technical_level      0
overtaking_level     0
lap_length_bucket    0
altitude_bucket      0
dtype: int64

HOME RACE FLAGS
--------------------------------------------------------------------------------
home_race_driver values:
home_race_driver
0    6679
1     232
Name: count, dtype: int64

home_race_constructor values:

### Analytical Supercheck

In [9]:
print("=" * 80)
print("SUPERCHECK — CIRCUIT FEATURES")
print("=" * 80)

print("\n1. CIRCUIT TYPE DISTRIBUTION")
print("-" * 80)
print(circuit_metadata["circuit_type"].value_counts(dropna=False))

print("\n2. DOWNFORCE LEVEL DISTRIBUTION")
print("-" * 80)
print(circuit_metadata["downforce_level"].value_counts(dropna=False))

print("\n3. SPEED PROFILE DISTRIBUTION")
print("-" * 80)
print(circuit_metadata["speed_profile"].value_counts(dropna=False))

print("\n4. TECHNICAL LEVEL DISTRIBUTION")
print("-" * 80)
print(circuit_metadata["technical_level"].value_counts(dropna=False))

print("\n5. ALTITUDE BUCKET DISTRIBUTION")
print("-" * 80)
print(circuit_metadata["altitude_bucket"].value_counts(dropna=False))

print("\n6. HIGHEST-DOWNFORCE CIRCUITS")
print("-" * 80)
display(
    circuit_metadata
    .sort_values(["downforce_score", "technical_score"], ascending=False)
    [[
        "circuit_id",
        "circuit_name",
        "circuit_type",
        "downforce_score",
        "speed_score",
        "technical_score",
        "overtaking_score",
        "lap_length_km",
        "altitude_m",
        "metadata_confidence"
    ]]
    .head(15)
)

print("\n7. HIGHEST-SPEED CIRCUITS")
print("-" * 80)
display(
    circuit_metadata
    .sort_values(["speed_score", "downforce_score"], ascending=[False, True])
    [[
        "circuit_id",
        "circuit_name",
        "circuit_type",
        "downforce_score",
        "speed_score",
        "technical_score",
        "overtaking_score",
        "lap_length_km",
        "altitude_m",
        "metadata_confidence"
    ]]
    .head(15)
)

print("\n8. HOME RACE SUMMARY")
print("-" * 80)
display(
    driver_race_with_circuit
    .query("home_race_driver == 1")
    .groupby(["driver_id", "driver_name"], as_index=False)
    .agg(home_races=("race_id", "nunique"))
    .sort_values("home_races", ascending=False)
    .head(25)
)

print("\n9. SOURCE TRANSPARENCY")
print("-" * 80)
print("Metadata confidence:")
print(circuit_metadata["metadata_confidence"].value_counts(dropna=False))
print("\nClassification source:")
print(circuit_metadata["classification_source"].value_counts(dropna=False))

print("\n10. SUPERCHECK VERDICT")
print("-" * 80)

supercheck_issues = []

if "monaco" in set(circuit_metadata["circuit_id"]):
    monaco = circuit_metadata.query("circuit_id == 'monaco'").iloc[0]
    if monaco["downforce_score"] < 8 or monaco["speed_score"] > 3:
        supercheck_issues.append("Monaco classification looks suspicious.")

if "monza" in set(circuit_metadata["circuit_id"]):
    monza = circuit_metadata.query("circuit_id == 'monza'").iloc[0]
    if monza["downforce_score"] > 3 or monza["speed_score"] < 8:
        supercheck_issues.append("Monza classification looks suspicious.")

if "rodriguez" in set(circuit_metadata["circuit_id"]):
    mexico = circuit_metadata.query("circuit_id == 'rodriguez'").iloc[0]
    if mexico["altitude_m"] < 1500:
        supercheck_issues.append("Mexico City altitude looks suspicious.")

if supercheck_issues:
    print("REVIEW NEEDED")
    for issue in supercheck_issues:
        print("-", issue)
else:
    print("PASS — circuit features look analytically reasonable.")

SUPERCHECK — CIRCUIT FEATURES

1. CIRCUIT TYPE DISTRIBUTION
--------------------------------------------------------------------------------
circuit_type
permanent    24
street        6
hybrid        5
Name: count, dtype: int64

2. DOWNFORCE LEVEL DISTRIBUTION
--------------------------------------------------------------------------------
downforce_level
medium    24
high       7
low        4
Name: count, dtype: int64

3. SPEED PROFILE DISTRIBUTION
--------------------------------------------------------------------------------
speed_profile
high_speed    17
balanced      15
low_speed      3
Name: count, dtype: int64

4. TECHNICAL LEVEL DISTRIBUTION
--------------------------------------------------------------------------------
technical_level
medium    19
high      16
Name: count, dtype: int64

5. ALTITUDE BUCKET DISTRIBUTION
--------------------------------------------------------------------------------
altitude_bucket
low       26
medium     7
high       2
Name: count, dtype: int

,circuit_id,circuit_name,circuit_type,downforce_score,speed_score,technical_score,overtaking_score,lap_length_km,altitude_m,metadata_confidence
14,monaco,Circuit de Monaco,street,10,1,10,1,3.337,10,high
12,marina_bay,Marina Bay Street Circuit,street,9,3,9,4,4.940,5,high
7,hungaroring,Hungaroring,permanent,9,3,8,3,4.381,264,high
26,suzuka,Suzuka Circuit,permanent,8,8,10,5,5.807,45,high
5,catalunya,Circuit de Barcelona-Catalunya,permanent,8,6,8,4,4.657,109,high
29,zandvoort,Circuit Zandvoort,permanent,8,5,8,3,4.259,5,high
21,rodriguez,Autodromo Hermanos Rodriguez,permanent,8,8,6,8,4.304,2240,high
8,imola,Autodromo Enzo e Dino Ferrari,permanent,7,7,8,4,4.909,37,medium
16,mugello,Mugello Circuit,permanent,7,8,8,4,5.245,292,medium
17,nurburgring,Nurburgring,permanent,7,6,8,5,5.148,620,medium



7. HIGHEST-SPEED CIRCUITS
--------------------------------------------------------------------------------


,circuit_id,circuit_name,circuit_type,downforce_score,speed_score,technical_score,overtaking_score,lap_length_km,altitude_m,metadata_confidence
15,monza,Autodromo Nazionale di Monza,permanent,1,10,4,8,5.793,162,high
34,vegas,Las Vegas Strip Circuit,street,2,10,5,7,6.201,620,high
3,baku,Baku City Circuit,street,3,10,7,7,6.003,-20,medium
11,jeddah,Jeddah Corniche Circuit,street,4,10,8,6,6.174,15,high
25,spa,Circuit de Spa-Francorchamps,permanent,5,10,9,7,7.004,400,high
20,ricard,Circuit Paul Ricard,permanent,4,9,5,5,5.842,432,medium
24,silverstone,Silverstone Circuit,permanent,7,9,8,6,5.891,153,high
30,losail,Lusail International Circuit,permanent,7,9,7,4,5.419,14,high
27,villeneuve,Circuit Gilles Villeneuve,hybrid,3,8,5,8,4.361,13,high
13,miami,Miami International Autodrome,hybrid,4,8,5,6,5.412,2,high



8. HOME RACE SUMMARY
--------------------------------------------------------------------------------


,driver_id,driver_name,home_races
1,alonso,Fernando Alonso,17
16,hamilton,Lewis Hamilton,17
42,sainz,Carlos Sainz,11
37,ricciardo,Daniel Ricciardo,10
50,vettel,Sebastian Vettel,9
34,perez,Sergio Pérez,9
26,massa,Felipe Massa,8
18,hulkenberg,Nico Hülkenberg,8
32,norris,Lando Norris,8
41,russell,George Russell,8



9. SOURCE TRANSPARENCY
--------------------------------------------------------------------------------
Metadata confidence:
metadata_confidence
high      21
medium    14
Name: count, dtype: int64

Classification source:
classification_source
analyst-coded estimate; not an official F1 numeric rating    35
Name: count, dtype: int64

10. SUPERCHECK VERDICT
--------------------------------------------------------------------------------
PASS — circuit features look analytically reasonable.


### Save outputs

In [11]:
circuit_metadata_path = EXTERNAL / "circuit_metadata.csv"
driver_circuit_path = PROCESSED / "driver_race_with_circuit_features.csv"

circuit_metadata.to_csv(circuit_metadata_path, index=False)
driver_race_with_circuit.to_csv(driver_circuit_path, index=False)

circuit_saved = pd.read_csv(circuit_metadata_path)
driver_saved = pd.read_csv(driver_circuit_path)

print("Saved circuit metadata:", circuit_metadata_path.exists())
print("Circuit rows match:", len(circuit_saved) == len(circuit_metadata))
print("Circuit columns match:", circuit_saved.shape[1] == circuit_metadata.shape[1])

print("\nSaved driver-race with circuit features:", driver_circuit_path.exists())
print("Driver rows match:", len(driver_saved) == len(driver_race_with_circuit))
print("Driver columns match:", driver_saved.shape[1] == driver_race_with_circuit.shape[1])

Saved circuit metadata: True
Circuit rows match: True
Circuit columns match: True

Saved driver-race with circuit features: True
Driver rows match: True
Driver columns match: True
